In [40]:
import numpy as np
import cv2 as cv
import os

# Load camera calibration data from previous calibration
calib_dir = 'Data/camera'
calib_file = os.path.join(calib_dir, 'calibration.npz')

if not os.path.exists(calib_file):
    print(f"ERROR: Calibration file not found: {calib_file}")
    print("Please run calibration.ipynb first.")
else:
    data = np.load(calib_file, allow_pickle=True)
    
    mtx = data['mtx']                           # Camera intrinsic matrix
    dist = data['dist']                         # Distortion coefficients
    rvecs = data['rvecs']                       # Camera rotation vectors (per image)
    tvecs = data['tvecs']                       # Camera translation vectors (per image)
    num_corners_x = int(data['num_interior_corners_x'])
    num_corners_y = int(data['num_interior_corners_y'])
    checker_width = float(data['checker_width'])
    
    print("✓ Loaded camera calibration data")
    print(f"  Camera Matrix:\n{mtx}")
    print(f"  Distortion: {dist.flatten()}")
    print(f"  Number of poses: {len(rvecs)}")
    print(f"  Checkerboard: {num_corners_x}×{num_corners_y} @ {checker_width} mm")

✓ Loaded camera calibration data
  Camera Matrix:
[[511.26900803   0.         331.82406479]
 [  0.         510.57392452 233.96461313]
 [  0.           0.           1.        ]]
  Distortion: [ 0.3111306  -3.53527523  0.01375121  0.01559911  8.77023405]
  Number of poses: 16
  Checkerboard: 9×6 @ 10.006 mm


In [41]:
# Reconstruct object points (checkerboard 3D coordinates)
objp = np.zeros((num_corners_x * num_corners_y, 3), np.float32)
objp[:, :2] = np.mgrid[0:num_corners_x, 0:num_corners_y].T.reshape(-1, 2)
objp = objp * checker_width

# Create list of object points (same for each image)
object_points = [objp for _ in range(len(rvecs))]

print(f"✓ Reconstructed {len(object_points)} object point sets")
print(f"  Checkerboard shape in 3D space: {objp.shape}")
print(f"  First point: {objp[0]} mm")
print(f"  Last point: {objp[-1]} mm")

✓ Reconstructed 16 object point sets
  Checkerboard shape in 3D space: (54, 3)
  First point: [0. 0. 0.] mm
  Last point: [80.048 50.03   0.   ] mm


In [60]:
# Load robot end-effector poses from probetobase*.txt files
import re
import glob

probe_dir = 'Data/probetobase'
probe_files = sorted(
    glob.glob(os.path.join(probe_dir, 'probetobase*.txt')),
    key=lambda x: int(re.search(r'(\d+)', os.path.basename(x)).group(1))
)

print(f"Found {len(probe_files)} robot pose files")

if len(probe_files) == 0:
    print("ERROR: No probetobase*.txt files found")
else:
    robot_poses = []

    for pose_file in probe_files:
        T_raw = np.loadtxt(pose_file, delimiter=',')
        if T_raw.shape != (4, 4):
            raise ValueError(f"{pose_file} does not contain a 4x4 matrix (got {T_raw.shape})")

        # print("Inverting pose to get T_base->gripper")
        # T_base2gripper = np.linalg.inv(T_raw)
        T_gripper2base = T_raw
        robot_poses.append(T_gripper2base)
        print(f"  {os.path.basename(pose_file)}: {T_gripper2base.shape}")

    robot_poses = np.array(robot_poses, dtype=float)
    print(f"\n✓ Loaded {len(robot_poses)} robot poses as T_gripper->base")
    print(f"  Shape: {robot_poses.shape}")
    print(f"  First pose:\n{robot_poses[0]}")

    # Extract rotation and translation from 4x4 matrices (R_base2gripper, t_base2gripper)
    robot_rvecs = []
    robot_tvecs = []

    for pose in robot_poses:
        R = pose[:3, :3]
        t = pose[:3, 3:4]
        R = cv.Rodrigues(R)[0]  # Convert rotation matrix to Rodrigues vector
        robot_rvecs.append(R)
        robot_tvecs.append(t)

    robot_rvecs = np.array(robot_rvecs, dtype=float)
    robot_tvecs = np.array(robot_tvecs, dtype=float)

    print(f"\n✓ Extracted base->gripper rotation/translation:")
    print(f"  Robot R list shape: {robot_rvecs.shape}")
    print(f"  Robot t list shape: {robot_tvecs.shape}")

Found 16 robot pose files
  probetobase1.txt: (4, 4)
  probetobase2.txt: (4, 4)
  probetobase3.txt: (4, 4)
  probetobase4.txt: (4, 4)
  probetobase5.txt: (4, 4)
  probetobase6.txt: (4, 4)
  probetobase7.txt: (4, 4)
  probetobase8.txt: (4, 4)
  probetobase9.txt: (4, 4)
  probetobase10.txt: (4, 4)
  probetobase11.txt: (4, 4)
  probetobase12.txt: (4, 4)
  probetobase13.txt: (4, 4)
  probetobase14.txt: (4, 4)
  probetobase15.txt: (4, 4)
  probetobase16.txt: (4, 4)

✓ Loaded 16 robot poses as T_gripper->base
  Shape: (16, 4, 4)
  First pose:
[[-7.11479000e-01 -7.02070000e-01  2.99150000e-02  1.56038586e+02]
 [-6.93779000e-01  6.95038000e-01 -1.88657000e-01 -3.09105091e+02]
 [ 1.11659000e-01 -1.54980000e-01 -9.81587000e-01  9.67258080e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]

✓ Extracted base->gripper rotation/translation:
  Robot R list shape: (16, 3, 1)
  Robot t list shape: (16, 3, 1)


In [ ]:
# Perform hand-eye calibration using cv.calibrateRobotWorldHandEye()
print("="*70)
print("PERFORMING HAND-EYE CALIBRATION")
print("="*70)

# Convert rvecs/tvecs to list format (these are camera-to-checkerboard = world-to-camera)
R_world2cam_list = [rvecs[i].reshape(3, 1) if rvecs[i].ndim == 1 else rvecs[i] 
                    for i in range(len(rvecs))]
t_world2cam_list = [tvecs[i].reshape(3, 1) if tvecs[i].ndim == 1 else tvecs[i] 
                    for i in range(len(tvecs))]

# # Robot poses: base to gripper
# R_base2gripper_list = [robot_rvecs[i].reshape(3, 1) if robot_rvecs[i].ndim == 1 else robot_rvecs[i] 
#                         for i in range(len(robot_rvecs))]
# t_base2gripper_list = [robot_tvecs[i].reshape(3, 1) if robot_tvecs[i].ndim == 1 else robot_tvecs[i] 
#                         for i in range(len(robot_tvecs))]

R_gripper2base_list = [robot_rvecs[i].reshape(3, 1) if robot_rvecs[i].ndim == 1 else robot_rvecs[i] 
                        for i in range(len(robot_rvecs))]
t_gripper2base_list = [robot_tvecs[i].reshape(3, 1) if robot_tvecs[i].ndim == 1 else robot_tvecs[i] 
                        for i in range(len(robot_tvecs))]

# Perform calibration - returns 4 outputs
R_base2world, t_base2world, R_gripper2cam, t_gripper2cam = cv.calibrateHandEye(
    R_gripper2base_list,   # Rotation: gripper to base
    t_gripper2base_list,   # Translation: gripper to base
    R_world2cam_list,      # Rotation: world (checkerboard) to camera
    t_world2cam_list,      # Translation: world to camera
)

print(f"✓ Hand-eye calibration successful!")
print(f"\nOutput transformations computed:")
print(f"  - R_base2world: Robot base to world frame (rotation)")
print(f"  - t_base2world: Robot base to world frame (translation)")
print(f"  - R_gripper2cam: Gripper to camera frame (rotation)")
print(f"  - t_gripper2cam: Gripper to camera frame (translation)\n")

# Display results
print("="*70)
print("HAND-EYE CALIBRATION RESULTS")
print("="*70)

print(f"\n1. Base to World transformation:")
print(f"   Rotation Matrix (R_base2world):")
print(R_base2world)
print(f"   Translation (t_base2world, mm):")
print(t_base2world.flatten())

print(f"\n2. Gripper to Camera transformation:")
print(f"   Rotation Matrix (R_gripper2cam):")
print(R_gripper2cam)
print(f"   Translation (t_gripper2cam, mm):")
print(t_gripper2cam.flatten())

# Convert to 4x4 transformation matrices for convenience
T_base2world = np.eye(4)
T_base2world[:3, :3] = R_base2world
T_base2world[:3, 3] = t_base2world.flatten()

T_gripper2cam = np.eye(4)
T_gripper2cam[:3, :3] = R_gripper2cam
T_gripper2cam[:3, 3] = t_gripper2cam.flatten()

print(f"\nBase to World - 4x4 Transformation Matrix:")
print(T_base2world)

print(f"\nGripper to Camera - 4x4 Transformation Matrix:")
print(T_gripper2cam)

# Save results
handeye_file = os.path.join(calib_dir, 'handeye_calibration.npz')
np.savez(handeye_file,
         R_base2world=R_base2world,
         t_base2world=t_base2world,
         T_base2world=T_base2world,
         R_gripper2cam=R_gripper2cam,
         t_gripper2cam=t_gripper2cam,
         T_gripper2cam=T_gripper2cam,
         mtx=mtx,
         dist=dist)

print(f"\n✓ Saved hand-eye calibration to: {handeye_file}")
print(f"\nSaved transformations:")
print(f"  - T_base2world: Transform points from robot base to world frame")
print(f"  - T_gripper2cam: Transform points from gripper to camera frame")

PERFORMING HAND-EYE CALIBRATION
✓ Hand-eye calibration successful!

Output transformations computed:
  - R_base2world: Robot base to world frame (rotation)
  - t_base2world: Robot base to world frame (translation)
  - R_gripper2cam: Gripper to camera frame (rotation)
  - t_gripper2cam: Gripper to camera frame (translation)

HAND-EYE CALIBRATION RESULTS

1. Base to World transformation:
   Rotation Matrix (R_base2world):
[[-0.57977489 -0.81411766  0.03276437]
 [-0.81477268  0.57942948 -0.02017324]
 [-0.00256125 -0.03839145 -0.99925949]]
   Translation (t_base2world, mm):
[-132.4487129  414.6096397  -44.0183058]

2. Gripper to Camera transformation:
   Rotation Matrix (R_gripper2cam):
[[ 0.99918094 -0.03245335 -0.02417097]
 [ 0.03227052  0.99944782 -0.00791601]
 [ 0.02441452  0.00712951  0.9996765 ]]
   Translation (t_gripper2cam, mm):
[ 24.32344988  81.53469611 157.94869453]

Base to World - 4x4 Transformation Matrix:
[[-5.79774891e-01 -8.14117664e-01  3.27643684e-02 -1.32448713e+02]
 [

In [58]:
# Loop-closure error check: world -> cam -> gripper -> base -> world
print("\n" + "="*70)
print("LOOP-CLOSURE ERROR CHECK")
print("="*70)

def make_T(R, t):
    T = np.eye(4, dtype=float)
    T[:3, :3] = np.asarray(R, dtype=float).reshape(3, 3)
    T[:3, 3] = np.asarray(t, dtype=float).reshape(-1)[:3]
    return T

def rotation_error_deg(R):
    value = (np.trace(R) - 1.0) / 2.0
    value = np.clip(value, -1.0, 1.0)
    return np.degrees(np.arccos(value))

def to_rotation_matrix(R_or_rvec):
    arr = np.asarray(R_or_rvec, dtype=float)
    # Supports either 3x3 rotation matrix or 3x1/1x3 rotation vector
    if arr.shape == (3, 3):
        return arr
    if arr.size == 3:
        Rm, _ = cv.Rodrigues(arr.reshape(3, 1))
        return Rm
    raise ValueError(f"Unsupported rotation format: shape={arr.shape}")

T_cam2gripper = np.linalg.inv(T_gripper2cam)

num_samples = min(len(rvecs), len(tvecs), len(robot_poses))
print(f"Using {num_samples} synchronized samples")

trans_errors_mm = []
rot_errors_deg = []

for i in range(num_samples):
    # world -> cam from checkerboard pose estimation
    R_wc = to_rotation_matrix(rvecs[i])
    t_wc = np.asarray(tvecs[i], dtype=float).reshape(-1)[:3]
    T_world2cam_i = make_T(R_wc, t_wc)

    # base -> gripper from robot logs
    T_base2gripper_i = np.asarray(robot_poses[i], dtype=float).reshape(4, 4)
    T_gripper2base_i = np.linalg.inv(T_base2gripper_i)

    # Loop: world -> cam -> gripper -> base -> world (should be identity)
    T_loop = T_world2cam_i @ T_cam2gripper @ T_gripper2base_i @ T_base2world

    trans_err = np.linalg.norm(T_loop[:3, 3])
    rot_err = rotation_error_deg(T_loop[:3, :3])

    trans_errors_mm.append(trans_err)
    rot_errors_deg.append(rot_err)

trans_errors_mm = np.array(trans_errors_mm)
rot_errors_deg = np.array(rot_errors_deg)

print("\nPer-sample loop errors:")
for i, (te, re) in enumerate(zip(trans_errors_mm, rot_errors_deg), start=1):
    print(f"  Sample {i:02d}: translation={te:8.3f} mm, rotation={re:7.4f} deg")

print("\nSummary:")
print(f"  Translation error mean/std/max: {trans_errors_mm.mean():.3f} / {trans_errors_mm.std():.3f} / {trans_errors_mm.max():.3f} mm")
print(f"  Rotation error mean/std/max:    {rot_errors_deg.mean():.4f} / {rot_errors_deg.std():.4f} / {rot_errors_deg.max():.4f} deg")

loop_metrics_file = os.path.join(calib_dir, 'handeye_loop_errors.npz')
np.savez(loop_metrics_file,
         trans_errors_mm=trans_errors_mm,
         rot_errors_deg=rot_errors_deg)
print(f"\nSaved loop metrics to: {loop_metrics_file}")


LOOP-CLOSURE ERROR CHECK
Using 16 synchronized samples

Per-sample loop errors:
  Sample 01: translation= 203.119 mm, rotation=25.0028 deg
  Sample 02: translation= 155.307 mm, rotation=34.2043 deg
  Sample 03: translation= 258.076 mm, rotation=26.8467 deg
  Sample 04: translation= 472.437 mm, rotation=32.3086 deg
  Sample 05: translation= 321.805 mm, rotation=34.9790 deg
  Sample 06: translation= 239.294 mm, rotation=49.6253 deg
  Sample 07: translation= 151.298 mm, rotation=78.9618 deg
  Sample 08: translation= 393.016 mm, rotation=52.7215 deg
  Sample 09: translation= 417.051 mm, rotation=46.5579 deg
  Sample 10: translation= 456.128 mm, rotation=60.6682 deg
  Sample 11: translation= 456.123 mm, rotation=60.6588 deg
  Sample 12: translation= 359.037 mm, rotation=41.8666 deg
  Sample 13: translation= 283.822 mm, rotation=46.6172 deg
  Sample 14: translation= 124.459 mm, rotation=79.8572 deg
  Sample 15: translation= 503.002 mm, rotation=77.5560 deg
  Sample 16: translation= 460.609 